In [1]:
import os
import sys

# Add the inference directory to the PYTHONPATH
som_path = '/scratch/bczf/zoezheng126/uouo/github-repo/UOUO/inference'
if som_path not in sys.path:
    sys.path.append(som_path)

os.environ['PYTHONPATH'] = os.environ.get('PYTHONPATH', '') + f":{som_path}"
os.environ['HF_HOME'] = '/scratch/bczf/zoezheng126/huggingface_cache'
print(os.getenv('HF_HOME'))

os.environ["OPENAI_API_KEY"] = 'sk-SDHmgyZBd5bPfRt0i1yj-HCEr83x-POtZA41NBmW_qT3BlbkFJHBmuErFwYQJaR7qvAAIxxfI770ICVH47-NZWDAS84A'

os.environ['PATH'] = '/sw/spack/deltas11-2023-03/apps/linux-rhel8-zen3/gcc-11.4.0/cuda-12.3.0-okhhaic/bin:' + os.environ['PATH']


/scratch/bczf/zoezheng126/huggingface_cache


In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Fri_Sep__8_19:17:24_PDT_2023
Cuda compilation tools, release 12.3, V12.3.52
Build cuda_12.3.r12.3/compiler.33281558_0


In [3]:
import os
import pickle
# from model import HFModelGeneration
from PIL import Image
from tqdm import tqdm
import time
import base64
import io
import glob
import torch

# need compile session

In [12]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
        return data

def resize_image(image, max_size):
    """Resize the image to ensure it's below the max size in bytes."""
    buffer = io.BytesIO()
    quality = 95
    while True:
        buffer.seek(0)
        buffer.truncate(0)
        image.save(buffer, format=image.format, quality=quality)
        size = buffer.tell()
        if size <= max_size or quality <= 10:
            break
        quality -= 5
    buffer.seek(0)
    return buffer

def encode_image(image_path, max_size=20 * 1024 * 1024):
    """Encode image to base64, resizing if necessary."""
    supported_formats = ['PNG', 'JPEG', 'GIF', 'WEBP']
    with Image.open(image_path) as image_file:
        if image_file.format not in supported_formats:
            raise ValueError(f"Unsupported image format: {image_file.format}. Supported formats are: {supported_formats}")

        buffer = resize_image(image_file, max_size) if os.path.getsize(image_path) > max_size else open(image_path, "rb")
        encoded_image = base64.b64encode(buffer.read()).decode('utf-8')
        buffer.close()
        return encoded_image
    
def save_output(output, output_dir, model_type, index):
    """Save the output dictionary to a pickle file."""
    os.makedirs(output_dir, exist_ok=True)
    output_filepath = os.path.join(output_dir, f'{model_type}_grounding_{index}.pkl')
    with open(output_filepath, 'wb') as file:
        pickle.dump(output, file)

def create_multi_image_prompt(object_name, gpt4_answer):
    template_1 = """
    This is a chat between a curious human and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the human's questions. If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don't know the answer to a question, please don't share false information. The assistant will have one similar in-context example provided by another powerful assistant of GPT4:
    Human:
    "Identify the location of the given object in this 2x2 mosaic image. The possible answers are: 'top left', 'top right', 'bottom left', 'bottom right', or 'none'. Only give a deterministic response as one of the possible answers. If the object is not present, the response should be 'none'. Please do not give more than one response.
    object name: {object_name}
    Location: "

    GPT4 Assistant:
    Choice: {gpt4_answer}
    """
    prompt_1 = template_1.format(
        gpt4_answer=gpt4_answer,
        object_name=object_name
    )

    template_2 = """
    Now you should answer the following question given the image below and you can use GPT4 Assistant's case for reference:
    Human:
    "Identify the location of the given object in this 2x2 mosaic image. The possible answers are: 'top left', 'top right', 'bottom left', 'bottom right', or 'none'. Only give a deterministic response as one of the possible answers. If the object is not present, the response should be 'none'. Please do not give more than one response. 
    
    Assistant(you):
    object name: {object_name}
    Location:
    """    
    prompt_2 = template_2.format(
        gpt4_answer=gpt4_answer,
        object_name=object_name
    )

    conversation_1 = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_1},
            {"type": "image"},
            {"type": "text", "text": prompt_2},
            ],
    }]
    return conversation_1

def create_single_image_prompt(object_name):
    template_2 = """
    Now you should answer the following question given the image below:
    Human:
    "Identify the location of the given object in this 2x2 mosaic image. The possible answers are: 'top left', 'top right', 'bottom left', 'bottom right', or 'none'. Only give a deterministic response as one of the possible answers. If the object is not present, the response should be 'none'. Please do not give more than one response.
    
    Assistant(you):
    object name: {object_name}
    Location:
    """    
    prompt_2 = template_2.format(
        object_name=object_name
    )

    conversation_1 = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_2},
            ],
    }]
    return conversation_1

def collect_files(directory):
    # Get the list of subdirectories sorted by numerical order
    subdirs = sorted([d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))], key=int)
    
    # Collect files in sorted order based on the subdirectory names
    files = []
    for subdir in subdirs:
        # Get the file under each numbered subdirectory
        subdir_path = os.path.join(directory, subdir)
        files_in_subdir = glob.glob(f'{subdir_path}/*')  # Assumes there is only one file in each subdir
        if files_in_subdir:
            files.append(files_in_subdir[0])  # Append the first file found in the subdirectory
    return files



In [ ]:
# Load the filename-to-category dictionary
with open('/scratch/bczf/zoezheng126/uouo/mosaic/preprocess/category_lookup.pkl', 'rb') as f:
    file_dict = pickle.load(f)

# small vlm mosaic dir
test_mosaic_dir = '/scratch/bczf/zoezheng126/uouo/mosaic/outputs/model-cascade-small-vlm-shuffle-2/Mosaic-Image'
# large vlm mosaic dir
gt_mosaic_dir = '/scratch/bczf/zoezheng126/uouo/mosaic/outputs/model-cascade-big-vlm-shuffle-2/Mosaic-Image'

# Prepare for processing
mosaics = collect_files(test_mosaic_dir)
gt_mosaics = collect_files(gt_mosaic_dir)
num_of_mosaics = len(mosaics)
print(f'There are {num_of_mosaics} mosaics in total.')

# cascade or no cascade from groundtruth example

In [5]:
# NEED TO MODIFY!!!!!!!!!!!!!!!!!!!!!

model_name = 'llava-hf/llava-v1.6-vicuna-7b-hf'
multi_gpu = False 
output_dir = '/scratch/bczf/zoezheng126/uouo/github-repo/UOUO/inference/outputs/llava-1.6-vicuna-7b-gpt-4o-cascade-new-prompt-shuffle'
test_mosaic_dir = '/scratch/bczf/zoezheng126/uouo/mosaic/outputs/model-cascade-small-vlm-shuffle-2/Mosaic-Image'
gt_mosaic_dir = '/scratch/bczf/zoezheng126/uouo/mosaic/outputs/model-cascade-big-vlm-shuffle-2/Mosaic-Image'

model_type = 'llava-v1.6-vicuna-7b-hf'

In [7]:
from transformers import AutoProcessor, LlavaNextForConditionalGeneration

model = LlavaNextForConditionalGeneration.from_pretrained("llava-hf/llava-v1.6-vicuna-7b-hf", torch_dtype=torch.float16, device_map="auto")
processor = AutoProcessor.from_pretrained("llava-hf/llava-v1.6-vicuna-7b-hf")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [8]:
# in use
def process_mosaic_model_cascade_single_model(mosaic_filename, gt_mosaic_filename, file_dict, image_dir, gt_image_dir, processor, model):
    """Process a single mosaic to generate responses."""
    # try:
    # process groundtruth filaname -> 4 category:
    gt_mosaic_id = gt_mosaic_filename.split('.')[0]
    gt_four_file_names = gt_mosaic_id.split('_')[1:]
    gt_four_cat_names = [file_dict[file] for file in gt_four_file_names]
    # process test fileanme -> 4 category
    mosaic_id = mosaic_filename.split('.')[0]
    four_file_names = mosaic_id.split('_')[1:]
    four_cat_names = [file_dict[file] for file in four_file_names]

    test_image_path = os.path.join(image_dir, mosaic_filename)
    print(f'test_image_path: {test_image_path}')
    test_image = Image.open(test_image_path).convert('RGB')

    gt_image_path = os.path.join(image_dir, gt_mosaic_filename)
    print(f'test_image_path: {gt_image_path}')
    gt_image = Image.open(gt_image_path).convert('RGB')

    ans_dict = {0:"top left", 1: "top right", 2: "bottom left", 3: "bottom right"}
    prompts = []

    responses_four = []
    for i, cat in enumerate(gt_four_cat_names):
        question = create_multi_image_prompt(cat,ans_dict[i])
        prompts = [processor.apply_chat_template(question, add_generation_prompt=True)]
        images = [gt_image, test_image]
        inputs = processor(text=prompts, images=images, padding=True, return_tensors="pt").to(model.device)
        # print(inputs)
        generate_ids = model.generate(**inputs, max_new_tokens=30)
        responses = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        print(responses)
        responses_four.append(responses)
    return {mosaic_id: dict(zip(gt_four_cat_names, responses_four))}
    # except Exception as e:
    #     print(f"Failed to process {mosaic_filename}: {e}")
    #     return None

def process_mosaic_single_model(mosaic_filename, gt_mosaic_filename, file_dict, image_dir, gt_image_dir, processor, model):
    """Process a single mosaic to generate responses."""
    # try:
    # process groundtruth filaname -> 4 category:
    gt_mosaic_id = gt_mosaic_filename.split('.')[0]
    gt_four_file_names = gt_mosaic_id.split('_')[1:]
    gt_four_cat_names = [file_dict[file] for file in gt_four_file_names]
    # process test fileanme -> 4 category
    mosaic_id = mosaic_filename.split('.')[0]
    four_file_names = mosaic_id.split('_')[1:]
    four_cat_names = [file_dict[file] for file in four_file_names]

    test_image_path = os.path.join(image_dir, mosaic_filename)
    print(f'test_image_path: {test_image_path}')
    test_image = Image.open(test_image_path).convert('RGB')

    gt_image_path = os.path.join(image_dir, gt_mosaic_filename)
    print(f'test_image_path: {gt_image_path}')
    gt_image = Image.open(gt_image_path).convert('RGB')

    ans_dict = {0:"top left", 1: "top right", 2: "bottom left", 3: "bottom right"}
    prompts = []

    responses_four = []
    for i, cat in enumerate(gt_four_cat_names):
        question = create_single_image_prompt(cat)
        prompts = [processor.apply_chat_template(question, add_generation_prompt=True)]
        images = [test_image]
        inputs = processor(text=prompts, images=images, padding=True, return_tensors="pt").to(model.device)
        # print(inputs)
        generate_ids = model.generate(**inputs, max_new_tokens=30)
        responses = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        print(responses)
        responses.append(ans_dict[i])
        responses_four.append(responses)
    return {mosaic_id: dict(zip(gt_four_cat_names, responses_four))}
    # except Exception as e:
    #     print(f"Failed to process {mosaic_filename}: {e}")
    #     return None

def process_mosaic_model_cascade_single_model_shuffle(mosaic_filename, gt_mosaic_filename, file_dict, image_dir, gt_image_dir, processor, model):
    """Process a single mosaic to generate responses."""
    # try:
    # process groundtruth filaname -> 4 category:
    gt_mosaic_id = gt_mosaic_filename.split('.')[0]
    gt_four_file_names = gt_mosaic_id.split('_')[1:]
    gt_four_cat_names = [file_dict[file] for file in gt_four_file_names]
    # process test fileanme -> 4 category
    mosaic_id = mosaic_filename.split('.')[0]
    four_file_names = mosaic_id.split('_')[1:]
    four_cat_names = [file_dict[file] for file in four_file_names]

    test_image_path = os.path.join(image_dir, mosaic_filename)
    print(f'test_image_path: {test_image_path}')
    test_image = Image.open(test_image_path).convert('RGB')

    gt_image_path = os.path.join(gt_image_dir, gt_mosaic_filename)
    print(f'test_image_path: {gt_image_path}')
    gt_image = Image.open(gt_image_path).convert('RGB')

    idx_dict = {0:"top left", 1: "top right", 2: "bottom left", 3: "bottom right"}
    ans_dict = dict()
    for i in range(len(gt_four_file_names)):
        ans_dict[gt_four_cat_names[i]] = idx_dict[i]
    prompts = []

    responses_four = []
    for i, cat in enumerate(four_cat_names):
        question = create_multi_image_prompt(cat,ans_dict[cat])
        prompts = [processor.apply_chat_template(question, add_generation_prompt=True)]
        images = [gt_image, test_image]
        inputs = processor(text=prompts, images=images, padding=True, return_tensors="pt").to(model.device)
        # print(inputs)
        generate_ids = model.generate(**inputs, max_new_tokens=30)
        responses = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        responses.append({'test image': mosaic_filename, 'gt image': gt_mosaic_filename, 'cat': cat})
        print(responses)
        responses_four.append(responses)
    return {mosaic_id: dict(zip(gt_four_cat_names, responses_four))}
    # except Exception as e:
    #     print(f"Failed to process {mosaic_filename}: {e}")
    #     return None

def process_mosaic_single_model_shuffle(mosaic_filename, gt_mosaic_filename, file_dict, image_dir, gt_image_dir, processor, model):
    """Process a single mosaic to generate responses."""
    # try:
    # process groundtruth filaname -> 4 category:
    gt_mosaic_id = gt_mosaic_filename.split('.')[0]
    gt_four_file_names = gt_mosaic_id.split('_')[1:]
    gt_four_cat_names = [file_dict[file] for file in gt_four_file_names]
    # process test fileanme -> 4 category
    mosaic_id = mosaic_filename.split('.')[0]
    four_file_names = mosaic_id.split('_')[1:]
    four_cat_names = [file_dict[file] for file in four_file_names]

    test_image_path = os.path.join(image_dir, mosaic_filename)
    print(f'test_image_path: {test_image_path}')
    test_image = Image.open(test_image_path).convert('RGB')

    gt_image_path = os.path.join(image_dir, gt_mosaic_filename)
    print(f'test_image_path: {gt_image_path}')
    gt_image = Image.open(gt_image_path).convert('RGB')

    idx_dict = {0:"top left", 1: "top right", 2: "bottom left", 3: "bottom right"}
    ans_dict = dict()
    for i in range(len(gt_four_file_names)):
        ans_dict[gt_four_cat_names[i]] = idx_dict[i]
    prompts = []

    responses_four = []
    for i, cat in enumerate(four_cat_names):
        question = create_single_image_prompt(cat)
        prompts = [processor.apply_chat_template(question, add_generation_prompt=True)]
        images = [test_image]
        inputs = processor(text=prompts, images=images, padding=True, return_tensors="pt").to(model.device)
        # print(inputs)
        generate_ids = model.generate(**inputs, max_new_tokens=30)
        responses = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        print(responses)
        responses.append({'test image': mosaic_filename, 'gt image': gt_mosaic_filename, 'cat': cat})
        responses_four.append(responses)
    return {mosaic_id: dict(zip(gt_four_cat_names, responses_four))}
    # except Exception as e:
    #     print(f"Failed to process {mosaic_filename}: {e}")
    #     return None

In [ ]:
# no shuffle and model cascade from gt
output = dict()
for i, mosaic_filename in enumerate(tqdm(mosaics)):
    test_mosaic_dir_w_index = os.path.join(test_mosaic_dir, str(i+1))
    gt_mosaic_dir_w_index = os.path.join(gt_mosaic_dir, str(i+1))

    print(f'mosaic_filename: {mosaic_filename}')
    print(f'gt_mosaics[i]: {gt_mosaics[i]}')
    result = process_mosaic_model_cascade_single_model(mosaic_filename, gt_mosaics[i], file_dict, test_mosaic_dir_w_index, gt_mosaic_dir_w_index, processor, model)
    
    if result:
        output.update(result)

    if (i % 100 == 0 and i != 0) or i == num_of_mosaics - 1:
        save_output(output, output_dir, model_type, i)
        output = dict()


In [ ]:
#no shuffle and no model cascade from gt
output = dict()
for i, mosaic_filename in enumerate(tqdm(mosaics)):
    test_mosaic_dir_w_index = os.path.join(test_mosaic_dir, str(i+1))
    gt_mosaic_dir_w_index = os.path.join(gt_mosaic_dir, str(i+1))

    print(f'mosaic_filename: {mosaic_filename}')
    print(f'gt_mosaics[i]: {gt_mosaics[i]}')
    result = process_mosaic_single_model(mosaic_filename, gt_mosaics[i], file_dict, test_mosaic_dir_w_index, gt_mosaic_dir_w_index, processor, model)
    
    if result:
        output.update(result)

    if (i % 100 == 0 and i != 0) or i == num_of_mosaics - 1:
        save_output(output, output_dir, model_type, i)
        output = dict()

In [ ]:
# shuffle and model cascade from gt
output = dict()
for i, mosaic_filename in enumerate(tqdm(mosaics)):
    test_mosaic_dir_w_index = os.path.join(test_mosaic_dir, str(i+1))
    gt_mosaic_dir_w_index = os.path.join(gt_mosaic_dir, str(i+1))

    print(f'mosaic_filename: {mosaic_filename}')
    print(f'gt_mosaics[i]: {gt_mosaics[i]}')
    result = process_mosaic_model_cascade_single_model_shuffle(mosaic_filename, gt_mosaics[i], file_dict, test_mosaic_dir_w_index, gt_mosaic_dir_w_index, processor, model)
    
    if result:
        output.update(result)

    if (i % 100 == 0 and i != 0) or i == num_of_mosaics - 1:
        save_output(output, output_dir, model_type, i)
        output = dict()


In [ ]:
# shuffle and no cascade from gt
output = dict()
for i, mosaic_filename in enumerate(tqdm(mosaics)):
    test_mosaic_dir_w_index = os.path.join(test_mosaic_dir, str(i+1))
    gt_mosaic_dir_w_index = os.path.join(gt_mosaic_dir, str(i+1))

    print(f'mosaic_filename: {mosaic_filename}')
    print(f'gt_mosaics[i]: {gt_mosaics[i]}')
    result = process_mosaic_single_model_shuffle(mosaic_filename, gt_mosaics[i], file_dict, test_mosaic_dir_w_index, gt_mosaic_dir_w_index, processor, model)
    
    if result:
        output.update(result)

    if (i % 100 == 0 and i != 0) or i == num_of_mosaics - 1:
        save_output(output, output_dir, model_type, i)
        output = dict()


# run gpt to get gpt answer

In [9]:
def create_single_image_prompt_gpt(object_name):
    template = """
    Now you should answer the following question given the image below:
    Human:
    "Identify the location of the given object in this 2x2 mosaic image. The possible answers are: 'top left', 'top right', 'bottom left', 'bottom right', or 'none'. Only give a deterministic response as one of the possible answers. If the object is not present, the response should be 'none'. Please do not give more than one response.
    
    Assistant(you):
    object name: {object_name}
    Location:
    """    
    prompt = template.format(
        object_name=object_name
    )
    return prompt

def process_mosaic_shuffle_gpt4o(mosaic_filename, file_dict, generation):
    """Process a single mosaic to generate responses."""
    # process test fileanme -> 4 category
    mosaic_id = mosaic_filename.split('.')[0]
    four_file_names = mosaic_id.split('_')[1:]
    four_cat_names = [file_dict[file] for file in four_file_names]

    test_image_path = mosaic_filename
    # print(f'test_image_path: {test_image_path}')
    test_image = encode_image(mosaic_filename)

    idx_dict = {0:"top left", 1: "top right", 2: "bottom left", 3: "bottom right"}
    responses_four = []

    for i, cat in enumerate(four_cat_names):
        question = create_single_image_prompt_gpt(cat)
        responses = generation.generate_response(test_image, question)
        responses['metadata']={'test image': mosaic_filename, 'cat': cat, 'gt_location': idx_dict[i]}
        responses_four.append(responses)
    return {mosaic_id: dict(zip(four_cat_names, responses_four))}

In [58]:
# setup
model_name = 'gpt-4o-mini'
multi_gpu = False
multi_gpu = False 
output_dir = '/scratch/bczf/zoezheng126/uouo/github-repo/UOUO/inference/outputs/gpt-4o-mini-ans-big-vlm-new-prompt-shuffle-2'
test_mosaic_dir = '/scratch/bczf/zoezheng126/uouo/mosaic/outputs/model-cascade-big-vlm-shuffle-2/Mosaic-Image' # the mosaic dir for gpt infer
model_type = model_name

# # Load the filename-to-category dictionary
with open('/scratch/bczf/zoezheng126/uouo/mosaic/preprocess/category_lookup.pkl', 'rb') as f:
    file_dict = pickle.load(f)

# Prepare for processing
mosaics = collect_files(test_mosaic_dir)
num_of_mosaics = len(mosaics)
print(f'There are {num_of_mosaics} mosaics in total.')


There are 409 mosaics in total.


In [ ]:
# setup gpt model
from model import HFModelGeneration
generation = HFModelGeneration()
generation.from_pretrained(model_name, multi_gpu=multi_gpu)
model_type = model_name.split("/")[1] if '/' in model_name else model_name

In [60]:
# generate gpt answer
output = dict()
for i, mosaic_filename in enumerate(tqdm(mosaics)):
    result = process_mosaic_shuffle_gpt4o(mosaic_filename, file_dict, generation)
    if result:
        output.update(result) 

    if (i % 100 == 0 and i != 0) or i == num_of_mosaics - 1:
        save_output(output, output_dir, model_type, i)
        output = dict()

  0%|          | 0/409 [00:00<?, ?it/s]

100%|██████████| 409/409 [29:04<00:00,  4.27s/it]


In [27]:
# gpt example
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')
  
base64_image = encode_image('/scratch/bczf/zoezheng126/uouo/mosaic/outputs/model-cascade-big-vlm-shuffle-2/Mosaic-Image/1/mo_003730025_160990016_169860035_016070027.png')
question = create_single_image_prompt_gpt('Ammonium sulfate plant')

In [ ]:
# gpt example
from openai import OpenAI
import requests
api_key = 'sk-SDHmgyZBd5bPfRt0i1yj-HCEr83x-POtZA41NBmW_qT3BlbkFJHBmuErFwYQJaR7qvAAIxxfI770ICVH47-NZWDAS84A'
headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {api_key}"
}

payload = {
  "model": "gpt-4o",
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "What’s in this image?"
        },
        {
          "type": "image_url",
          "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image}"
          }
        }
      ]
    }
  ],
  "max_tokens": 300
}

response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)

print(response.json())

# gpt cascade to llava-7b

In [ ]:
# NEED TO MODIFY!!!!!!!!!!!!!!!!!!!!!

model_name = 'llava-hf/llava-v1.6-vicuna-7b-hf'
multi_gpu = False 
output_dir = '/scratch/bczf/zoezheng126/uouo/github-repo/UOUO/inference/outputs/llava-1.6-vicuna-7b-gpt-4o-cascade-new-prompt-shuffle'
test_mosaic_dir = '/scratch/bczf/zoezheng126/uouo/mosaic/outputs/model-cascade-small-vlm-shuffle-2/Mosaic-Image' # small vlm infer use
gt_mosaic_dir = '/scratch/bczf/zoezheng126/uouo/mosaic/outputs/model-cascade-big-vlm-shuffle-2/Mosaic-Image' # big vlm infer use

model_type = 'llava-v1.6-vicuna-7b-hf'

In [ ]:
from transformers import AutoProcessor, LlavaNextForConditionalGeneration

model = LlavaNextForConditionalGeneration.from_pretrained("llava-hf/llava-v1.6-vicuna-7b-hf", torch_dtype=torch.float16, device_map="auto")
processor = AutoProcessor.from_pretrained("llava-hf/llava-v1.6-vicuna-7b-hf")

In [10]:
# gpt4o cascade to llava 7b start from here

def process_mosaic_gpt_cascade_single_model_shuffle(mosaic_filename, gt_mosaic_filename, file_dict, gpt_ans, image_dir, gt_image_dir, processor, model):
    """Process a single mosaic to generate responses."""
    # try:
    # process groundtruth filaname -> 4 category:
    gt_mosaic_id = gt_mosaic_filename.split('.')[0]
    gt_four_file_names = gt_mosaic_id.split('_')[1:]
    gt_four_cat_names = [file_dict[file] for file in gt_four_file_names]
    # process test fileanme -> 4 category
    mosaic_id = mosaic_filename.split('.')[0]
    four_file_names = mosaic_id.split('_')[1:]
    four_cat_names = [file_dict[file] for file in four_file_names]

    test_image_path = os.path.join(image_dir, mosaic_filename)
    test_image = Image.open(test_image_path).convert('RGB')

    gt_image_path = os.path.join(gt_image_dir, gt_mosaic_filename)
    gt_image = Image.open(gt_image_path).convert('RGB')

    prompts = []

    responses_four = []
    for i, cat in enumerate(four_cat_names):
        if cat.lower() not in gpt_ans[gt_mosaic_id].keys():
            gpt_cat_ans = 'none'
        else:
            gpt_cat_ans = gpt_ans[gt_mosaic_id][cat.lower()]
        question = create_multi_image_prompt(cat,gpt_cat_ans)
        prompts = [processor.apply_chat_template(question, add_generation_prompt=True)]
        images = [gt_image, test_image]
        inputs = processor(text=prompts, images=images, padding=True, return_tensors="pt").to(model.device)
        # print(inputs)
        generate_ids = model.generate(**inputs, max_new_tokens=30)
        responses = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        responses.append({'test image': mosaic_filename, 'gt image': gt_mosaic_filename, 'cat': cat})
        responses_four.append(responses)
    return {mosaic_id: dict(zip(gt_four_cat_names, responses_four))}
    # except Exception as e:
    #     print(f"Failed to process {mosaic_filename}: {e}")
    #     return None

In [31]:
print(output_dir)

/scratch/bczf/zoezheng126/uouo/github-repo/UOUO/inference/outputs/llava-1.6-vicuna-7b-gpt-4o-cascade-new-prompt-shuffle


In [11]:
# shuffle model cascade from gpt-4o
path_to_gpt_ans = '/scratch/bczf/zoezheng126/uouo/github-repo/UOUO/inference/postprocess/gpt-4o-mosaic-shuffle-2-ans.pkl'
output = dict()
gpt_ans = load_pickle(path_to_gpt_ans)
for i, mosaic_filename in enumerate(tqdm(mosaics)):
    test_mosaic_dir_w_index = os.path.join(test_mosaic_dir, str(i+1))
    gt_mosaic_dir_w_index = os.path.join(gt_mosaic_dir, str(i+1))

    result = process_mosaic_gpt_cascade_single_model_shuffle(mosaic_filename, gt_mosaics[i], file_dict, gpt_ans, test_mosaic_dir_w_index, gt_mosaic_dir_w_index, processor, model)
    
    if result:
        output.update(result)

    if (i % 100 == 0 and i != 0) or i == num_of_mosaics - 1:
        save_output(output, output_dir, model_type, i)
        output = dict()


  0%|          | 0/409 [00:00<?, ?it/s]

Expanding inputs for image tokens in LLaVa-NeXT should be done in processing. Please add `patch_size` and `vision_feature_select_strategy` to the model's processing config or set directly with `processor.patch_size = {{patch_size}}` and processor.vision_feature_select_strategy = {{vision_feature_select_strategy}}`. Using processors without these attributes in the config is deprecated and will throw an error in v4.47.
Expanding inputs for image tokens in LLaVa-NeXT should be done in processing. Please add `patch_size` and `vision_feature_select_strategy` to the model's processing config or set directly with `processor.patch_size = {{patch_size}}` and processor.vision_feature_select_strategy = {{vision_feature_select_strategy}}`. Using processors without these attributes in the config is deprecated and will throw an error in v4.47.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
100%|██████████| 409/